In [125]:
import torch as t
import outlines
from transformers import AutoTokenizer, AutoModelForCausalLM
from pydantic import BaseModel
from typing import Literal
from enum import Enum
import yaml
import pandas as pd
import numpy as np
import sys
import os
import json
from openai import OpenAI
sys.path.append("../")
from src.utils import inverse_likert, list_to_str
device = t.device("cuda" if t.cuda.is_available() else "cpu")

In [37]:
class OpenaiResponse(BaseModel):
    response: str

In [115]:
with open('../configs/generation_config.yaml', 'r') as file:
    generation_config = yaml.safe_load(file)
    
with open('../psychometric_tests/hexaco_100_questions.yaml', 'r') as file:
    question_list = yaml.safe_load(file)
    
with open('../psychometric_tests/paraphrased_hexaco_100_questions.yaml', 'r') as file:
    paraphrased_question_list = yaml.safe_load(file)
    
with open('../psychometric_tests/hexaco_100_eval.yaml', 'r') as file:
    hexaco_eval = yaml.safe_load(file)

In [5]:
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
model = outlines.from_transformers(
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map=device),
    AutoTokenizer.from_pretrained(MODEL_NAME)
)

In [6]:
openai_model_name = "gpt-4.1-mini"
openai_model = outlines.from_openai(OpenAI(), openai_model_name)

In [7]:
NO_ANSWER = "Do not wish to answer"
likert_scale = generation_config['likert_scale'].copy()
likert_scale.append(NO_ANSWER)

In [8]:
inverted_likert = inverse_likert(generation_config['likert_scale'].copy())
inverted_likert.append(NO_ANSWER)

In [9]:
hexaco_template = outlines.Template.from_string("""
<|im_start>user
Task: Answer the below questions:

{{ text }}

Answer the question as either {{ likert_scale }}.
<|im_end>
<|im_start>assistant
""")


In [28]:
print(prompt)

<|im_start>user
Task: Answer the below questions:

I would be quite bored by a visit to an art gallery.

Answer the question as either strongly disagree, disagree, neutral, agree, strongly agree.
<|im_end>
<|im_start>assistant


In [10]:
likert_scale

['Strongly Disagree',
 'Disagree',
 'Neutral',
 'Agree',
 'Strongly Agree',
 'Do not wish to answer']

In [45]:
def ollama_generation(model, question, likert_scale):
    prompt = hexaco_template(text=question, likert_scale = ", ".join(likert_scale))
    answer = model(
                    prompt,
                    Literal[*likert_scale]
            )
    return answer

def openai_generation(model, question, likert_scale):
    prompt = hexaco_template(text=question, likert_scale = ", ".join(likert_scale))
    prompt = f"{prompt}, use the json format."
    
    answer = openai_model(prompt, OpenaiResponse)
    return json.loads(answer)['response']
    

def generate_answers(generation_function, model, question_list, likert_scale):
    
    answers = []
    for question in question_list:
        answer = generation_function(model, question, likert_scale)
        answers.append(answer)
    
    return answers

In [83]:
def write_to_json(file, file_path):
    with open(file_path, 'w') as f:
        json.dump(file, f)
        
def read_json(file_path):
    with open(file_path, "r") as f:
        file = json.load(f)
    return file

In [46]:
print(prompt)

<|im_start>user
Task: Answer the below questions:

I would be quite bored by a visit to an art gallery.

Answer the question as either Strongly Disagree, Disagree, Neutral, Agree, Strongly Agree, Do not wish to answer.
<|im_end>
<|im_start>assistant


### Normal Questions, Normal Likert

In [47]:
normal_hexaco_answers_gpt_41_mini = generate_answers(openai_generation, openai_model, question_list, likert_scale)

In [61]:
write_to_json(normal_hexaco_answers_gpt_41_mini, "normal_hexaco_answers_gpt_41_mini.json")

In [48]:
normal_hexaco_answers_llama_3_2b_it = generate_answers(ollama_generation, model, question_list, likert_scale)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [62]:
write_to_json(normal_hexaco_answers_llama_3_2b_it, "normal_hexaco_answers_llama_3_2b_it.json")

### Normal Questions, Inverse_Likert

In [54]:
normal_hexaco_inverted_likert_answers_gpt_41_mini = generate_answers(openai_generation, openai_model, question_list, inverted_likert)

In [63]:
write_to_json(normal_hexaco_inverted_likert_answers_gpt_41_mini, "normal_hexaco_inverted_likert_answers_gpt_41_mini.json")

In [55]:
normal_hexaco_inverted_likert_answers_llama_3_2b_it = generate_answers(ollama_generation, model, question_list, inverted_likert)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [64]:
write_to_json(normal_hexaco_inverted_likert_answers_llama_3_2b_it, "normal_hexaco_inverted_likert_answers_llama_3_2b_it.json")

### Paraphrase Questions, Normal Likert

In [56]:
paraphrase_hexaco_answers_gpt_41_mini = generate_answers(openai_generation, openai_model, paraphrased_question_list, likert_scale)

In [65]:
write_to_json(paraphrase_hexaco_answers_gpt_41_mini, "paraphrase_hexaco_answers_gpt_41_mini.json")

In [57]:
paraphrase_hexaco_answers_llama_3_2b_it = generate_answers(ollama_generation, model, paraphrased_question_list, likert_scale)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [66]:
write_to_json(paraphrase_hexaco_answers_llama_3_2b_it, "paraphrase_hexaco_answers_llama_3_2b_it.json")

### Paraphrase Questions, Inverted Likert

In [58]:
paraphrase_hexaco_inverted_likert_answers_gpt_41_mini = generate_answers(openai_generation, openai_model, paraphrased_question_list, inverted_likert)

In [67]:
write_to_json(paraphrase_hexaco_inverted_likert_answers_gpt_41_mini, "paraphrase_hexaco_inverted_likert_answers_gpt_41_mini.json")

In [59]:
paraphrase_hexaco_inverted_likert_answers_llama_3_2b_it = generate_answers(ollama_generation, model, paraphrased_question_list, inverted_likert)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [68]:
write_to_json(paraphrase_hexaco_inverted_likert_answers_llama_3_2b_it, "paraphrase_hexaco_inverted_likert_answers_llama_3_2b_it.json")

### Evaluation

In [86]:
def get_refusal_rate(answers):  
    answers = pd.Series(answers)
    refusal_answers = answers[answers == "Do not wish to answer"]
    return len(refusal_answers)/len(answers)

In [102]:
refusal_rate_dict = {}
for filename in os.listdir("base_experiment_results"):
    answers = read_json(os.path.join("base_experiment_results",filename))
    refusal_rate = get_refusal_rate(answers)
    refusal_rate_dict[filename.split(".")[0]] = refusal_rate

In [103]:
dict(sorted(refusal_rate_dict.items()))

{'normal_hexaco_answers_gpt_41_mini': 0.01,
 'normal_hexaco_answers_llama_3_2b_it': 0.09,
 'normal_hexaco_inverted_likert_answers_gpt_41_mini': 0.02,
 'normal_hexaco_inverted_likert_answers_llama_3_2b_it': 0.11,
 'paraphrase_hexaco_answers_gpt_41_mini': 0.02,
 'paraphrase_hexaco_answers_llama_3_2b_it': 0.14,
 'paraphrase_hexaco_inverted_likert_answers_gpt_41_mini': 0.0,
 'paraphrase_hexaco_inverted_likert_answers_llama_3_2b_it': 0.16}

In [289]:
def calculate_hexaco_score(trait, subtrait, answers, likert_scale = generation_config['likert_scale']):
    answer_dict = {}
    answers = pd.Series(answers)
    subtrait_dict = hexaco_eval[trait][subtrait]
    indices = [idx - 1 for idx in subtrait_dict['indices']]
    trait_answers = answers[indices]
    refused_answers = trait_answers[trait_answers == 'Do not wish to answer']
    non_refused_answers = trait_answers[trait_answers != 'Do not wish to answer']
    answer_indices = [likert_scale.index(answer) for answer in non_refused_answers]
    true_answer_indices = [6-idx if reverse else idx for idx,reverse in zip(answer_indices, subtrait_dict['reverse'])]
    
    answer_dict['answer_indices'] = answer_indices
    answer_dict['true_answer_indices'] = true_answer_indices
    answer_dict['n_answered_questions'] = len(true_answer_indices)
    answer_dict['n_refused_questions'] = len(refused_answers)
    answer_dict['trait'] = trait
    answer_dict['subtrait'] = subtrait
    answer_dict['subtrait_score'] = np.round(np.mean(true_answer_indices).item(),2)
    return answer_dict

In [290]:
def get_all_stats(filename, hexaco_eval, likert_scale=generation_config['likert_scale']):
    trait_hexaco_scores = []
    answers = read_json(os.path.join("base_experiment_results",filename))
    for trait in hexaco_eval.keys():
        for subtrait in hexaco_eval[trait].keys():
            hexaco_score = calculate_hexaco_score(trait, subtrait, answers, likert_scale)
            if "inverted" in filename:
                hexaco_score['likert_scale'] = "inverse"
            else:
                hexaco_score['likert_scale'] = "normal"
            if "paraphrase" in filename:
                hexaco_score['paraphrase'] = "paraphrase"
            else:
                hexaco_score['paraphrase'] = "normal"
            if "llama" in filename:
                hexaco_score['model'] = "llama_3.2_1b_it"
            else:
                hexaco_score['model'] = "gpt_4.1_mini"
                
            trait_hexaco_scores.append(hexaco_score)
    return trait_hexaco_scores

In [291]:
stat_df = pd.DataFrame()
for filename in os.listdir("base_experiment_results"):
    df = pd.DataFrame(get_all_stats(filename, hexaco_eval))
    stat_df = pd.concat([stat_df,df], axis = 0)
stat_df.reset_index(inplace=True, drop=True)

In [292]:
gpt_stat_df = stat_df[stat_df['model'] == "gpt_4.1_mini"]
llama_stat_df = stat_df[stat_df['model'] == "llama_3.2_1b_it"]

In [293]:
gpt_stat_df.loc[:,'refusal_rate'] = gpt_stat_df.loc[:,'n_refused_questions']/(gpt_stat_df.loc[:,'n_answered_questions'] + gpt_stat_df.loc[:,'n_refused_questions'])
llama_stat_df.loc[:,'refusal_rate'] = llama_stat_df.loc[:,'n_refused_questions']/(llama_stat_df.loc[:,'n_answered_questions'] + llama_stat_df.loc[:,'n_refused_questions'])

/var/folders/9b/k67tngbx13jgzw5kjx9_d0tc0000gn/T/ipykernel_27949/2074838846.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gpt_stat_df.loc[:,'refusal_rate'] = gpt_stat_df.loc[:,'n_refused_questions']/(gpt_stat_df.loc[:,'n_answered_questions'] + gpt_stat_df.loc[:,'n_refused_questions'])
/var/folders/9b/k67tngbx13jgzw5kjx9_d0tc0000gn/T/ipykernel_27949/2074838846.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  llama_stat_df.loc[:,'refusal_rate'] = llama_stat_df.loc[:,'n_refused_questions']/(llama_stat

In [294]:
gpt_stat_df.pivot(values = ['refusal_rate','subtrait_score',], columns=['paraphrase', 'likert_scale'], index=['trait','subtrait'])

refusal_rate                  \
paraphrase                                      paraphrase          normal   
likert_scale                                        normal inverse inverse   
trait                  subtrait                                              
agreeableness          flexibility                     0.0     0.0    0.00   
                       forgiveness                     0.0     0.0    0.00   
                       gentleness                      0.0     0.0    0.00   
                       patience                        0.0     0.0    0.00   
altruism               altruism                        0.0     0.0    0.00   
conscientiousness      diligence                       0.0     0.0    0.00   
                       organization                    0.0     0.0    0.00   
                       perfectionism                   0.0     0.0    0.00   
                       prudence                        0.0     0.0    0.00   
emotionality           anxiety                         0.0     0.0    0.00   
                       dependence                      0.0     0.0    0.00   
                       fearfulness                     0.0     0.0    0.00   
                       sentimentality                  0.0     0.0    0.00   
extraversion           liveliness                      0.0     0.0    0.00   
                       sociability                     0.0     0.0    0.00   
                       social boldness                 0.0     0.0    0.00   
                       social self-esteem              0.0     0.0    0.00   
honest-humility        fairness                        0.5     0.0    0.00   
                       greed-avoidance                 0.0     0.0    0.00   
                       modesty                         0.0     0.0    0.25   
                       sincerity                       0.0     0.0    0.25   
openness to experience aesthetic appreciation          0.0     0.0    0.00   
                       creativity                      0.0     0.0    0.00   
                       inquisitiveness                 0.0     0.0    0.00   
                       unconventionality               0.0     0.0    0.00   

                                                     subtrait_score          \
paraphrase                                               paraphrase           
likert_scale                                  normal         normal inverse   
trait                  subtrait                                               
agreeableness          flexibility              0.00           3.50    3.50   
                       forgiveness              0.00           3.00    3.00   
                       gentleness               0.00           3.25    3.25   
                       patience                 0.00           3.25    3.00   
altruism               altruism                 0.00           4.25    4.25   
conscientiousness      diligence                0.00           3.50    3.50   
                       organization             0.00           3.25    3.25   
                       perfectionism            0.00           3.25    3.75   
                       prudence                 0.00           3.25    3.25   
emotionality           anxiety                  0.00           3.00    3.00   
                       dependence               0.00           3.00    3.00   
                       fearfulness              0.00           3.25    3.25   
                       sentimentality           0.00           3.25    3.25   
extraversion           liveliness               0.00           3.50    3.50   
                       sociability              0.00           3.00    2.75   
                       social boldness          0.00           3.25    3.00   
                       social self-esteem       0.00           3.25    3.25   
honest-humility        fairness                 0.25           3.50    4.75   
                       greed-avoidance       

In [295]:
llama_stat_df.pivot(values = ['refusal_rate','subtrait_score',], columns=['paraphrase', 'likert_scale'], index=['trait','subtrait'])

refusal_rate                 \
paraphrase                                      paraphrase  normal          
likert_scale                                        normal inverse normal   
trait                  subtrait                                             
agreeableness          flexibility                    0.50    0.00   0.00   
                       forgiveness                    0.00    0.00   0.00   
                       gentleness                     0.00    0.00   0.00   
                       patience                       0.00    0.00   0.00   
altruism               altruism                       0.00    0.25   0.00   
conscientiousness      diligence                      0.25    0.50   0.25   
                       organization                   0.75    0.25   0.50   
                       perfectionism                  0.50    0.00   0.00   
                       prudence                       0.00    0.00   0.00   
emotionality           anxiety                        0.00    0.25   0.25   
                       dependence                     0.00    0.25   0.00   
                       fearfulness                    0.00    0.25   0.00   
                       sentimentality                 0.00    0.00   0.25   
extraversion           liveliness                     0.25    0.00   0.00   
                       sociability                    0.00    0.25   0.00   
                       social boldness                0.00    0.00   0.00   
                       social self-esteem             0.00    0.00   0.00   
honest-humility        fairness                       0.25    0.00   0.25   
                       greed-avoidance                0.25    0.00   0.00   
                       modesty                        0.00    0.25   0.25   
                       sincerity                      0.00    0.25   0.00   
openness to experience aesthetic appreciation         0.00    0.00   0.00   
                       creativity                     0.00    0.00   0.00   
                       inquisitiveness                0.50    0.00   0.50   
                       unconventionality              0.25    0.25   0.00   

                                                         subtrait_score  \
paraphrase                                    paraphrase     paraphrase   
likert_scale                                     inverse         normal   
trait                  subtrait                                           
agreeableness          flexibility                  0.00           4.00   
                       forgiveness                  0.25           2.50   
                       gentleness                   0.75           3.00   
                       patience                     0.00           2.00   
altruism               altruism                     0.00           2.50   
conscientiousness      diligence                    0.25           3.33   
                       organization                 0.00           2.00   
                       perfectionism                0.00           3.00   
                       prudence                     0.00           3.50   
emotionality           anxiety                      0.25           3.00   
                       dependence                   0.25           2.50   
                       fearfulness                  0.25           2.00   
                       sentimentality               0.00           3.25   
extraversion           liveliness                   0.25           2.00   
                       sociability                  0.00           1.00   
                       social boldness              0.00           5.00   
                       social self-esteem           0.00           3.50   
honest-humility        fairness                     0.00           3.00   
                       greed-avoidance              0.75           1.33   
                       modesty                      0.25           3.25   
            